<!-- # <span style="color:red">UNDER CONSTRUCTION!!!!</span> -->

# Spoken Language Processing - Instituto Superior Técnico
### Laboratory Assignment 2 - Automatic Age Estimation Challenge
<!--[image](imgs/lab2_slp_banner.png)-->
<img src="imgs/lab2_slp_banner.png" alt="drawing" width="400"/>

# WEEK 2 - Using pre-trained models


During this week, students will implement two modern systems for age regression based on:
- speaker representations (utterance-based) obtained with an x-vector model (this notebook);
- speech representations (frame-based) obtained with a self-supervised learning (SSL) pre-trained model (`lab2_ssl.ipynb` notebook).

In both cases, students are encouraged to explore different feature configurations and alternative downstream models.

## Before starting

Let's import some modules and make some definitions. 

**WARNING from professors** We changed the pf_tools.py script for this second week. Be sure to update (the new one is compatible with Week 1 lab)

In [18]:
import os
import csv
import pickle
import numpy as np
import librosa
import torch

from pf_tools import CheckThisCell, SLPdata
from speechbrain.inference.classifiers import EncoderClassifier
from speechbrain.utils.data_utils import split_path
from sklearn.svm import LinearSVC, SVR
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt


GENDER_CLASSES = ('F',  'M')
GEN2ID = {'F':0, 'M':1}
ID2GEN = dict((GEN2ID[k],k)for k in GEN2ID)

Like in the previous Notebooks, you need to mount Google drive if you are working on Google Colab. Otherwise, you should skip or delete the following code cell:


Like in week1, the audio data is expected to be in a folder with the following format:

```
lab2_data/
├── train/
│   └── wav/
│       └──wav files
│   └── info.csv
│
└── train_small/
    └── wav/
        └──wav files
    └── info.csv
...
```

You must already have this from the previous week, so you can set-up your data directory:

In [19]:
import os

CWD = os.getcwd() 
DATADIR = os.path.abspath(os.path.join(CWD, '..', 'data'))

# Create the data directory if it doesn't exist
if not os.path.isdir(DATADIR):
    os.makedirs(DATADIR)
    print(f"Created directory: {DATADIR}")

print(f'Current working directory is set to: {CWD}')   
print(f'Your LAB2 data folder is: {DATADIR}')

Current working directory is set to: /home/luis/projects/slp_lab2/src
Your LAB2 data folder is: /home/luis/projects/slp_lab2/data


## Using pre-trained speaker embeddings (x-vectors)

The goal of this part of the lab is to become familiar with and show how to use pre-trained speaker embedings (a.k.a. x-vectors) for speech classification/regressions tasks.

There exist plenty of resources and pre-trained models that can be  useful for our task. In particular, x-vectors are the current state of the art approach to obtain speech embeddings that characterize very efficiently speaker or language, among others. X-vectors are neural models typically trained for speaker identification in a supervised way, but also in some cases for other related tasks. Once trained, they can be used to obtain a single embedding vector of fixed dimension for each audio input. This vector corresponds to the activations of one of the layers after the pooling layer.

The following are examples of x-vector models available in the `speechbrain` module:

- `speechbrain/spkrec-xvect-voxceleb`: same with a different architecture: https://huggingface.co/speechbrain/spkrec-xvect-voxceleb

- `speechbrain/spkrec-ecapa-voxceleb`: trained using a large speaker corpus for speaker verification: https://huggingface.co/speechbrain/spkrec-ecapa-voxceleb



The following code cell shows how to import one of those models to obtain an embedding vector:

In [20]:
# Instantiate the model. If you don't have GPU available, run this line
# xvector_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-xvect-voxceleb", savedir=f"{CWD}/tmp")

# If you have GPU available, run this line instead
xvector_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-xvect-voxceleb", 
    savedir=f"{CWD}/tmp_new", 
    run_opts={"device":"cuda:0"}
)

signal = xvector_model.load_audio(f'{DATADIR}/train/wav/00834c0e904d40eda496e55010acebc5.wav')
emb =  xvector_model.encode_batch(signal)  

print(type(emb), emb.shape)

<class 'torch.Tensor'> torch.Size([1, 1, 512])


These (very informative) embedding vectors can be used to train simple models for several speech classification tasks, achieving excellent results. In particular, in this lab assignment, we will train a simple Support Vector Regression (SVR) on top of these x-vectors.


Student groups will be graded depending on their ability to explore different feature configurations and model alternatives/configurations.

### 1. Extracting x-vectors for the SLP datasets

Just like in Part1, we will code the feature transformation to process all data and obtain x-vectors. In this case, the function should receive as arguments the audio filename and an instance of `EncoderClassifier` (the x-vector model) and return the numpy array with the features. You must complete the following code using the previous example:

In [21]:
def extract_xvec(filename, emb_model):
    """
    Extract x-vector embedding from an audio file.
    Returns numpy array of shape (1, D).
    """
    # Load audio using the model's own loader (handles resampling)
    signal = emb_model.load_audio(filename)          # (T,) tensor
 
    # encode_batch expects (batch, T) — unsqueeze adds batch dim
    embedding = emb_model.encode_batch(signal.unsqueeze(0))  # (1, 1, D)
 
    # Squeeze to (1, D) and move to CPU numpy
    embedding = embedding.squeeze(1).detach().cpu().numpy()  # (1, D)
 
    # Remove the soft-link created by speechbrain
    _, fl = split_path(filename)
    if os.path.islink(fl):
        os.remove(fl)
 
    return embedding   # shape (1, D)
 
 
# Quick sanity check
emb = extract_xvec(
    f'{DATADIR}/train/wav/00834c0e904d40eda496e55010acebc5.wav',
    xvector_model
)
print(emb.shape, type(emb))   # expect (1, 512) <class 'numpy.ndarray'>


(1, 512) <class 'numpy.ndarray'>


In [22]:
import os
import numpy as np
from speechbrain.inference.classifiers import EncoderClassifier
import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 1. LOAD BOTH MODELS EXACTLY ONCE
print("Loading SpeechBrain X-Vector...")
xvect_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-xvect-voxceleb", 
    savedir=f"{CWD}/tmp_xvect", 
    run_opts={"device": str(device)} 
)

print("Loading SpeechBrain ECAPA-TDNN...")
ecapa_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb", 
    savedir=f"{CWD}/tmp_ecapa", 
    run_opts={"device": str(device)} 
)

# 3. DEFINE THE COMBINED EXTRACTION FUNCTION
def extract_combined_xvect_ecapa(filename):
    # Extract from both models
    feat_xvect = extract_xvec(filename, xvect_model) # 512 dims
    feat_ecapa = extract_xvec(filename, ecapa_model) # 192 dims
    
    # Concatenate along the feature axis to get 704 dims
    return np.concatenate((feat_xvect, feat_ecapa), axis=1)

print("Transform correctly registered. Combined features will be 704 dimensions (512 + 192).")

Loading SpeechBrain X-Vector...
Loading SpeechBrain ECAPA-TDNN...
Transform correctly registered. Combined features will be 704 dimensions (512 + 192).


Let's generate the x-vectors for all our data sets using the SLP class and store in disk. Like in Part 1, we can keep different transformation configurations in a dictionary for later usage.  

Let's define first our configurations (you can try to different x-vector models, the propsed ones or even other that you may find in huggingface):

In [23]:
transform = {
    'spkrec-ecapa-voxceleb': {
        # THE FIX: Pass the pre-loaded ecapa_model here
        'audio_transform': lambda x: extract_xvec(x, emb_model=ecapa_model), 
        'chunk_transform': None,
        'chunk_size': 0,
        'chunk_hop': 0
    }
}

transform['spkrec-xvect-voxceleb'] = {
    # THE FIX: Pass the pre-loaded xvect_model here
    'audio_transform': lambda x: extract_xvec(x, emb_model=xvect_model), 
    'chunk_transform': None,
    'chunk_size': 0,
    'chunk_hop': 0
}

transform['combined-xvect-ecapa'] = {
    'audio_transform': extract_combined_xvect_ecapa,
    'chunk_transform': None,
    'chunk_size': 0,
    'chunk_hop': 0
}

And now let's do feature extraction. Be patient because this process can be a bit slow depending on the resources of your machine (train_small without GPU should take around 5min on google colab):

In [31]:
# Download and feature extract
trainset = 'train' # Change to 'train' when you are ready for the full run!
transform_id = 'spkrec-ecapa-voxceleb' # Change to 'combined-xvect-ecapa' or 'spkrec-ecapa-voxceleb'

slp_partitions = {}
# for partition in ('train', 'train_small', 'dev', 'evl'):
for partition in ('train', 'dev', 'evl'):
    print(f" -> Processing {partition}...")
    slp_partitions[partition] = SLPdata(
        DATADIR, partition,
        transform_id=transform_id,
        audio_transform=transform[transform_id]['audio_transform'],
        chunk_transform=transform[transform_id]['chunk_transform'],
        chunk_size=transform[transform_id]['chunk_size'],
        chunk_hop=transform[transform_id]['chunk_hop']
    )

 -> Processing train...


3238it [01:14, 43.48it/s]


 -> Processing dev...


117it [00:02, 46.34it/s]


 -> Processing evl...


145it [00:03, 47.14it/s]


### 2. Training an SVR model

Our first attempt of age regression system based on x-vectors will be a simple SVR model like in the `openSMILE` baseline, but in this case we will be using x-vectors as features.

First, we will use the SLP data instances to store the x-vectors, the labels and file identifiers in numpy arrays:

In [26]:
from pf_tools import prepare_slp_data

#   Concatenate all data and labels
#   Each row corresponds to a file
#   We store the data, labels and file identifiers of each partition in dictionaries
#     with the partition name as key


gender_label_pos = 0
age_label_pos = 1

data, labels_gender, labels_age, fileids = {}, {}, {}, {}
# for partition in ('train', 'train_small', 'dev', 'evl'):
for partition in ('train', 'dev', 'evl'):
    data_and_labels = prepare_slp_data(slp_partitions[partition])
    print(f'Partition: {partition}')
    print(f'Number of samples: {data_and_labels["data"].shape[0]}')
    print(f'Number of features: {data_and_labels["data"].shape[1]}')
    print(f'Number of labels: {len(np.unique(data_and_labels["label"][:,gender_label_pos]))}')
    print(f'Number of identifiers (samples): {len(np.unique(data_and_labels["identifiers"]))}')
    print('---')
    data[partition] = data_and_labels['data']
    labels_gender[partition] = data_and_labels['label'][:,gender_label_pos]
    labels_age[partition] = data_and_labels['label'][:,age_label_pos]
    fileids[partition] = data_and_labels['identifiers']



Partition: train
Number of samples: 3238
Number of features: 512
Number of labels: 2
Number of identifiers (samples): 3238
---
Partition: dev
Number of samples: 117
Number of features: 512
Number of labels: 2
Number of identifiers (samples): 117
---
Partition: evl
Number of samples: 145
Number of features: 512
Number of labels: 1
Number of identifiers (samples): 145
---


Now, we will use `sklearn` Support Vector Regression (SVR) to:
1. Train our regressor and save it for later use.
2. Predict on the dev and evl partitions and save the results

In [27]:
from sklearn.svm import LinearSVR
from pf_tools import save_model
import time


trainset = 'train'

print(f"Training LinearSVR on {data[trainset].shape[0]} samples, {data[trainset].shape[1]} features")
print(f"Started at: {time.strftime('%H:%M:%S')}")
t0 = time.time()

model = LinearSVR(max_iter=5000, verbose=1)
model.fit(data[trainset], labels_age[trainset])

print(f"Training done in {(time.time()-t0)/60:.1f} min")

model_id = save_model(model, f'svr_{transform_id}', f'{DATADIR}/{trainset}/models/')
print(f'Model {model_id} saved in {DATADIR}/{trainset}/models/')

t0 = time.time()
print("Predicting dev...")
dev_results = model.predict(data['dev'])
filename = f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl'
pickle.dump({'hyp': dev_results, 'fileids': fileids['dev']}, open(filename, 'wb'))
print(f"Dev done in {(time.time()-t0):.1f}s")

t0 = time.time()
print("Predicting evl...")
evl_results = model.predict(data['evl'])
filename = f'{DATADIR}/{trainset}/models/{model_id}/evl.pkl'
pickle.dump({'hyp': evl_results, 'fileids': fileids['evl']}, open(filename, 'wb'))
print(f"Evl done in {(time.time()-t0):.1f}s")

Training LinearSVR on 3238 samples, 512 features
Started at: 17:31:28
[LibLinear]....................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................
optimization finished, #iter = 5000

Using -s 11 may be faster

Objective value = -8119.095967
nSV = 3238
Training done in 0.2 min
Model saved to /home/luis/projects/slp_lab2/data/train/models//svr_spkrec-xvect-voxceleb_2026-05-22_17:31:40/model.pkl
Model svr_spkrec-xvect-voxceleb_2026-05-22_17:31:40 saved in /home/luis/projects/slp_lab2/data/train/models/
Predicting dev...
Dev done in 0.0s
Predicting evl...
Evl do

/home/luis/projects/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


It should be extremely easy to experiment other models provided in the `sklearn` module, including SVMs with other kernels, Random Forests, etc.


#### 2.1 Analyze results on the dev set and prepare your submission file

Let's check our performance on the dev set:

In [28]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

ref, hyp = labels_age['dev'], dev_results

print(f'Mean Absolute Error: {mean_absolute_error(ref, hyp):.2f}')
print(f'Mean Squared Error: {mean_squared_error(ref, hyp):.2f}')

Mean Absolute Error: 5.94
Mean Squared Error: 51.47


You should obtain a mean absolute error around 7.6 (it will depend on the xvector model chosen). 

Now, let's generate the final prediction file and make a submission to the  [Kaggle competition](https://www.kaggle.com/t/8d80747e0c474688a83024aabdfe1ab0):

In [ ]:
from pf_tools import create_submission_file

students_group = '00' # <--- CHANGE THIS ACCORDINGLY

model_id = 'svr_spkrec-xvect-voxceleb_2026-05-14_01:46:15'
model_id_short = 'svr_spkrec-xvect'

results_path = f'{DATADIR}/{trainset}/models/{model_id}/'
filename = f'{CWD}/g{students_group}_{trainset}_{model_id_short}.csv' # <--- CHANGE THIS ACCORDINGLY

create_submission_file(results_path, filename)


At this point, you can explore different x-vector model configurations for feature extraction and alternative models to the SVR.

### 3. Training a neural network model

As an alternative to the SVR, we will explore simple neural models on top of x-vector features.

We will need to define the size of the feature vector that will be used as input to the neural network:

In [29]:
feat_dim = data[trainset].shape[1]
print(f"transform_id : {transform_id}")
print(f"feat_dim     : {feat_dim}")

transform_id : spkrec-xvect-voxceleb
feat_dim     : 512


In [18]:
# ── Define aliases expected by the cleaning cell ──────────────────
X_tr = data[trainset]          # shape (N_train, feat_dim)
X_dv = data['dev']             # shape (N_dev,   feat_dim)
y_tr = labels_age[trainset]    # age targets for train
y_dv = labels_age['dev']       # age targets for dev

# ── Then your existing cleaning cell works as-is ──────────────────
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

imputer = SimpleImputer(strategy='mean')
scaler  = StandardScaler()

X_tr_clean = scaler.fit_transform(imputer.fit_transform(X_tr))
X_dv_clean = scaler.transform(imputer.transform(X_dv))
X_ev_clean = scaler.transform(imputer.transform(data['evl']))

print(f"After cleaning:")
print(f"X_train  NaN: {np.isnan(X_tr_clean).sum()}  max={X_tr_clean.max():.3f}")
print(f"X_dev    NaN: {np.isnan(X_dv_clean).sum()}  max={X_dv_clean.max():.3f}")

After cleaning:
X_train  NaN: 0  max=6.788
X_dev    NaN: 0  max=5.471


And a simple neural model architecture (you can change this):

We will use a simple `train_nn` function included in the `pf_tools` script that will permit training the model using backpropagation. Students are encouraged to explore this function and, eventually, to modify it to experiment alternative training strategies, parameters, etc.

In [30]:
# ================================================================
# STRATIFIED CV GRID SEARCH & BLIND DEV EVALUATION (XVEC RAW DATA)
# WITH RIDGE REGRESSOR BASELINE
# ================================================================

import itertools
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline
import pickle

# ── 1. Strict Reproducibility ────────────────────────────────────
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── 2. Data Preparation & Stratification ─────────────────────────
# Using the raw dictionaries directly from the extraction loop
X_full = data['train']
y_full = labels_age['train'].astype(float)

X_dev = data['dev']
y_dev_np = labels_age['dev'].astype(float)

X_dev_t = torch.FloatTensor(X_dev).to(device)

# Automatically determine the input dimension for the Neural Network
feat_dim = X_full.shape[1] 

# Create Age Bins for Stratification (e.g., <20, 20s, 30s, 40s, 50s, 60s+)
bins = [20, 30, 40, 50, 60, 70]
y_binned = np.digitize(y_full, bins)

N_SPLITS  = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

print(f"\nInput Feature Dimension: {feat_dim}")

# ── 3. Elastic Net Regressor Baseline ────────────────────────────
print(f"\n{'='*75}\n RUNNING ELASTIC NET BASELINE\n{'='*75}")
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error
import numpy as np

en_cv_maes = []
en_dev_preds = np.zeros((N_SPLITS, len(y_dev_np)))

# ---> THE FIX: Mute the Convergence Warnings completely
warnings.filterwarnings("ignore", category=ConvergenceWarning)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full, y_binned)):
    X_tr_f, y_tr_f = X_full[train_idx], y_full[train_idx]
    X_va_f, y_va_f = X_full[val_idx], y_full[val_idx]
    
    # Auto-tuning both L1 and L2
    model = make_pipeline(
        RobustScaler(),
        ElasticNetCV(
            l1_ratio=[0.001, 0.001, 0.01, 0.1, 0.5], 
            alphas=[0.001, 0.001, 0.01, 0.1, 1.0, 10.0],
            max_iter=10000,   # Set back to a reasonable time limit
            tol=1e-2,         # Tells the math engine "close enough is fine"
            n_jobs=-1, 
            random_state=42
        )
    )
    model.fit(X_tr_f, y_tr_f)
    
    val_hyp = model.predict(X_va_f)
    en_cv_maes.append(mean_absolute_error(y_va_f, val_hyp))
    
    en_dev_preds[fold, :] = np.clip(model.predict(X_dev), 18, 80)

    best_alpha = model.named_steps['elasticnetcv'].alpha_
    best_l1 = model.named_steps['elasticnetcv'].l1_ratio_
    print(f"Fold {fold + 1} | CV MAE: {en_cv_maes[-1]:.3f} | Alpha: {best_alpha} | L1 Ratio: {best_l1}")

en_mean_cv = np.mean(en_cv_maes)
en_blind_dev = mean_absolute_error(y_dev_np, np.mean(en_dev_preds, axis=0))
print(f"-> Elastic Net Baseline CV MAE: {en_mean_cv:.3f} | Blind Dev MAE: {en_blind_dev:.3f}\n")

# Turn warnings back on for the rest of your notebook
warnings.filterwarnings("default", category=ConvergenceWarning)


# ── 3.1 Ridge Regressor Baseline ──────────────────────────────────
print(f"\n{'='*75}\n RUNNING RIDGE REGRESSOR BASELINE\n{'='*75}")
ridge_cv_maes = []
ridge_dev_preds = np.zeros((N_SPLITS, len(y_dev_np)))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full, y_binned)):
    X_tr_f, y_tr_f = X_full[train_idx], y_full[train_idx]
    X_va_f, y_va_f = X_full[val_idx], y_full[val_idx]
    
    # Auto-scaling and auto-tuning L2 regularization
    from sklearn.decomposition import PCA

# This scales the data, compresses it to the 128 most important features, then runs Ridge
    model = make_pipeline(
        RobustScaler(),
        #PCA(n_components=512), 
        RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0], scoring='neg_mean_absolute_error')
    )
    model.fit(X_tr_f, y_tr_f)
    
    val_hyp = model.predict(X_va_f)
    ridge_cv_maes.append(mean_absolute_error(y_va_f, val_hyp))
    
    # Blind Dev Prediction
    ridge_dev_preds[fold, :] = np.clip(model.predict(X_dev), 18, 80)

best_alpha = model.named_steps['ridgecv'].alpha_
print(f"Fold {fold + 1} picked alpha: {best_alpha}")
ridge_mean_cv = np.mean(ridge_cv_maes)
ridge_blind_dev = mean_absolute_error(y_dev_np, np.mean(ridge_dev_preds, axis=0))
print(f"-> Ridge Baseline CV MAE: {ridge_mean_cv:.3f} | Blind Dev MAE: {ridge_blind_dev:.3f}\n")


# ── 4. PyTorch MLP Search Space ──────────────────────────────────
search_space = {
    'architecture': [
        [128, 64],
        [128, 128, 64],
        [256, 128, 64],
    ],
    'dropout':      [0.2, 0.5],
    'lr':           [1e-3, 1e-4],
    'weight_decay': [1e-3, 1e-4],
    'huber_delta':  [5.0, 10.0],
    'batch_size':   [32, 64],
}

keys   = list(search_space.keys())
all_combos = list(itertools.product(*search_space.values()))

MAX_SEARCH_TRIALS = 48  # Tests up to 48 random combinations
random.seed(42)
random.shuffle(all_combos)
combos_to_test = all_combos[:MAX_SEARCH_TRIALS]
N_TRIALS = len(combos_to_test)

MAX_EPOCH = 100
PATIENCE  = 25

print(f"{'='*75}\n STARTING MLP RANDOM SEARCH ({N_TRIALS} combinations)\n{'='*75}")

# ── 5. Model Architecture ────────────────────────────────────────
class ResBlockLN(nn.Module):
    def __init__(self, in_dim, out_dim, dropout):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(in_dim), nn.Linear(in_dim, out_dim), nn.GELU(),
            nn.Dropout(dropout), nn.LayerNorm(out_dim), nn.Linear(out_dim, out_dim), nn.GELU()
        )
        self.proj = nn.Linear(in_dim, out_dim, bias=False) if in_dim != out_dim else nn.Identity()
    def forward(self, x): return self.block(x) + self.proj(x)

def build_model(input_dim, arch, dropout):
    layers = [nn.Linear(input_dim, arch[0]), nn.GELU(), nn.Dropout(dropout)]
    for i in range(len(arch) - 1): layers.append(ResBlockLN(arch[i], arch[i + 1], dropout))
    layers.append(nn.Linear(arch[-1], 1))
    net = nn.Sequential(*layers)
    for m in net.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None: nn.init.zeros_(m.bias)
    return net

# ── 6. Grid Search Loop ──────────────────────────────────────────
results = []
total_start = time.time()

for t_idx, combo in enumerate(combos_to_test):
    cfg = dict(zip(keys, combo))
    arch, dropout, lr, wd, delta, bs = (
        cfg['architecture'], cfg['dropout'], cfg['lr'],
        cfg['weight_decay'], cfg['huber_delta'], cfg['batch_size']
    )
    
    fold_maes = []
    dev_preds_all_folds = np.zeros((N_SPLITS, len(y_dev_np)))
    
    # Run the 5 Folds
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_full, y_binned)):
        set_seed(42) # Fair initialization per fold
        
        X_tr_f = torch.FloatTensor(X_full[train_idx])
        y_tr_f = torch.FloatTensor(y_full[train_idx]).unsqueeze(1)
        X_val_f = torch.FloatTensor(X_full[val_idx]).to(device)
        y_val_f = y_full[val_idx]
        
        loader = DataLoader(TensorDataset(X_tr_f, y_tr_f), batch_size=bs, shuffle=True, drop_last=True)
        
        net = build_model(feat_dim, arch, dropout).to(device)
        opt = optim.AdamW(net.parameters(), lr=lr, weight_decay=wd)
        sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCH)
        crit = nn.HuberLoss(delta=delta)
        
        best_fold_mae = float('inf')
        best_state = None
        patience_cnt = 0
        
        for epoch in range(1, MAX_EPOCH + 1):
            net.train()
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                loss = crit(net(xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), 5.0)
                opt.step()
            sch.step()
            
            # Validation Step
            net.eval()
            with torch.no_grad():
                hyp = net(X_val_f).clamp(18, 80).cpu().numpy().ravel()
            mae = mean_absolute_error(y_val_f, hyp)
            
            if mae < best_fold_mae:
                best_fold_mae = mae
                best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}
                patience_cnt = 0
            else:
                patience_cnt += 1
                
            if patience_cnt >= PATIENCE: break
                
        fold_maes.append(best_fold_mae)
        
        # BLIND DEV PREDICTION
        net.load_state_dict(best_state)
        net.eval()
        with torch.no_grad():
            dev_preds_all_folds[fold, :] = net(X_dev_t).clamp(18, 80).cpu().numpy().ravel()

    # Summarize this configuration
    mean_cv_mae = np.mean(fold_maes)
    ensemble_dev_preds = np.mean(dev_preds_all_folds, axis=0)
    blind_dev_mae = mean_absolute_error(y_dev_np, ensemble_dev_preds)
    
    results.append({
        'cfg': cfg,
        'cv_mae': mean_cv_mae,
        'dev_mae': blind_dev_mae
    })
    
    print(f"[{t_idx + 1:>2}/{N_TRIALS}] CV_MAE: {mean_cv_mae:.3f} | Arch: {str(arch):<18} | Drop: {dropout} | LR: {lr} | WD: {wd} | Delta: {delta}")

# ── 7. Final Leaderboard (Testing the Best on Dev) ───────────────
results.sort(key=lambda x: x['cv_mae'])
total_min = (time.time() - total_start) / 60

print(f"\n{'='*75}")
print(f" GRID SEARCH COMPLETE — {total_min:.1f} min")
print(f"{'='*75}")
print(" FINAL LEADERBOARD:")
print(f"{'='*75}")

# Print Ridge First
print(f"[BASELINE] RIDGE REGRESSOR | CV MAE: {ridge_mean_cv:.3f} years --> BLIND DEV MAE: {ridge_blind_dev:.3f} years")
print("-" * 75)

# Print Top 5 MLPs
for rank, res in enumerate(results[:5], 1):
    c = res['cfg']
    print(f"[MLP #{rank}] CV MAE: {res['cv_mae']:.3f} years --> BLIND DEV MAE: {res['dev_mae']:.3f} years")
    print(f"          Arch: {c['architecture']}, Drop: {c['dropout']}, LR: {c['lr']}, WD: {c['weight_decay']}, Delta: {c['huber_delta']}\n")

Using device: cuda

Input Feature Dimension: 512

 RUNNING ELASTIC NET BASELINE
Fold 1 | CV MAE: 5.924 | Alpha: 0.01 | L1 Ratio: 0.001
Fold 2 | CV MAE: 6.143 | Alpha: 0.01 | L1 Ratio: 0.001
Fold 3 | CV MAE: 6.172 | Alpha: 0.01 | L1 Ratio: 0.001
Fold 4 | CV MAE: 6.250 | Alpha: 0.01 | L1 Ratio: 0.1
Fold 5 | CV MAE: 6.082 | Alpha: 0.01 | L1 Ratio: 0.5
-> Elastic Net Baseline CV MAE: 6.114 | Blind Dev MAE: 6.265


 RUNNING RIDGE REGRESSOR BASELINE
Fold 5 picked alpha: 10.0
-> Ridge Baseline CV MAE: 6.107 | Blind Dev MAE: 6.151

 STARTING MLP RANDOM SEARCH (48 combinations)


KeyboardInterrupt: 

In [8]:
# ================================================================
# QUICK TEST: 1D-CNN Regressor (5-Fold Stratified CV)
# ================================================================

import time
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import pickle

# ── 1. Strict Reproducibility ────────────────────────────────────
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── 2. Data Preparation & Scaling ────────────────────────────────
print("Preparing and Scaling Data...")
# Pulling directly from your extraction dictionaries
X_full_raw = data['train']
y_full = labels_age['train'].astype(float)
X_dev_raw  = data['dev']
y_dev_np   = labels_age['dev'].astype(float)

# Scaling is CRITICAL for Convolutional Networks
scaler = StandardScaler()
X_full = scaler.fit_transform(X_full_raw)
X_dev  = scaler.transform(X_dev_raw)

feat_dim = X_full.shape[1] 
print(f"Input Feature Dimension: {feat_dim}")

# Prepare Dev Tensor (Notice we DO NOT unsqueeze here, the model handles it)
X_dev_t = torch.FloatTensor(X_dev).to(device)

# Stratification Bins
bins = [20, 30, 40, 50, 60, 70]
y_binned = np.digitize(y_full, bins)

# ── 3. The 1D-CNN Architecture ───────────────────────────────────
class CNN1D_Regressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        
        # Convolutional Feature Extractor
        self.conv_block = nn.Sequential(
            # Layer 1
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.GELU(),
            nn.MaxPool1d(kernel_size=2), # Halves the spatial dimension
            
            # Layer 2
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.MaxPool1d(kernel_size=2)  # Halves the spatial dimension again
        )
        
        # Calculate Flattened Size: 32 channels * (input_dim / 4)
        linear_input = 32 * (input_dim // 4) 
        
        # Standard MLP Regressor Head
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(linear_input, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        # x arrives as shape (Batch, 704).
        # Conv1D expects (Batch, Channels, Length). 
        # We unsqueeze to make it (Batch, 1, 704)
        x = x.unsqueeze(1) 
        x = self.conv_block(x)
        x = self.regressor(x)
        return x

# ── 4. Training Loop ─────────────────────────────────────────────
N_SPLITS = 5
MAX_EPOCH = 80
PATIENCE  = 20
BATCH_SIZE = 64

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

cv_maes = []
dev_preds_all_folds = np.zeros((N_SPLITS, len(y_dev_np)))

print(f"\n{'='*75}\n STARTING 1D-CNN TRAINING (5 Folds)\n{'='*75}")
total_start = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full, y_binned)):
    set_seed(42) 
    
    # Create Tensors
    X_tr_f = torch.FloatTensor(X_full[train_idx])
    y_tr_f = torch.FloatTensor(y_full[train_idx]).unsqueeze(1)
    X_val_f = torch.FloatTensor(X_full[val_idx]).to(device)
    y_val_f = y_full[val_idx]
    
    loader = DataLoader(TensorDataset(X_tr_f, y_tr_f), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    
    # Init Model, Optimizer, Loss
    net = CNN1D_Regressor(feat_dim).to(device)
    opt = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-3)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCH)
    crit = nn.HuberLoss(delta=5.0)
    
    best_fold_mae = float('inf')
    best_state = None
    patience_cnt = 0
    
    for epoch in range(1, MAX_EPOCH + 1):
        net.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(net(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            opt.step()
        sch.step()
        
        # Validation
        net.eval()
        with torch.no_grad():
            hyp = net(X_val_f).clamp(18, 80).cpu().numpy().ravel()
        mae = mean_absolute_error(y_val_f, hyp)
        
        if mae < best_fold_mae:
            best_fold_mae = mae
            best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            
        if patience_cnt >= PATIENCE: 
            break
            
    cv_maes.append(best_fold_mae)
    
    # Save Dev Predictions for this fold
    net.load_state_dict(best_state)
    net.eval()
    with torch.no_grad():
        dev_preds_all_folds[fold, :] = net(X_dev_t).clamp(18, 80).cpu().numpy().ravel()
        
    print(f"Fold {fold + 1}/5 | Best CV MAE: {best_fold_mae:.3f} years (Stopped at Epoch {epoch})")

# ── 5. Evaluation ────────────────────────────────────────────────
total_min = (time.time() - total_start) / 60
mean_cv_mae = np.mean(cv_maes)

ensemble_dev_preds = np.mean(dev_preds_all_folds, axis=0)
blind_dev_mae = mean_absolute_error(y_dev_np, ensemble_dev_preds)

print(f"\n{'='*75}\n TRAINING COMPLETE — {total_min:.1f} min\n{'='*75}")
print(f"1D-CNN Mean CV MAE : {mean_cv_mae:.3f} (±{np.std(cv_maes):.3f})")
print(f"1D-CNN Blind Dev MAE: {blind_dev_mae:.3f} years")

Using device: cuda
Preparing and Scaling Data...
Input Feature Dimension: 704

 STARTING 1D-CNN TRAINING (5 Folds)
Fold 1/5 | Best CV MAE: 5.677 years (Stopped at Epoch 80)
Fold 2/5 | Best CV MAE: 5.720 years (Stopped at Epoch 80)
Fold 3/5 | Best CV MAE: 5.691 years (Stopped at Epoch 80)
Fold 4/5 | Best CV MAE: 5.583 years (Stopped at Epoch 80)
Fold 5/5 | Best CV MAE: 5.620 years (Stopped at Epoch 80)

 TRAINING COMPLETE — 3.1 min
1D-CNN Mean CV MAE : 5.659 (±0.050)
1D-CNN Blind Dev MAE: 6.766 years


In [50]:
# ── 8. Retrain and Save the Ridge Regressor Baseline ─────────────
print(f"\n{'='*75}")
print(" RETRAINING & SAVING RIDGE REGRESSOR ON FULL DATA")
print(f"{'='*75}")

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error
from pf_tools import save_model
import pickle
import numpy as np

# 1. Build the Pipeline 
# (RidgeCV will efficiently auto-tune the best alpha on the full set)
ridge_final = make_pipeline(
    RobustScaler(),
    RidgeCV(
        alphas=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0], 
        scoring='neg_mean_absolute_error'
    )
)

# 2. Train on 100% of the training data
print("Training on the full train set...")
# We use the raw dictionaries from memory, just like in your CV loop
ridge_final.fit(data['train'], labels_age['train'].astype(float))

best_alpha_final = ridge_final.named_steps['ridgecv'].alpha_
print(f"Final model selected alpha: {best_alpha_final}")

# 3. Save the model using the lab's built-in tool
model_id = save_model(ridge_final, f'ridge_baseline_{transform_id}', f'{DATADIR}/{trainset}/models/')
print(f"Model saved as: {model_id}")

# 4. Predict on Dev and Evl sets
print("Predicting on Dev and Evl sets...")
# We clip predictions between 18 and 80 just like you did in the CV loop
hyp_dev = np.clip(ridge_final.predict(data['dev']), 18, 80)
hyp_evl = np.clip(ridge_final.predict(data['evl']), 18, 80)

# 5. Save predictions to disk for your Kaggle submission
pickle.dump({'hyp': hyp_dev, 'fileids': fileids['dev']},
            open(f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl', 'wb'))
pickle.dump({'hyp': hyp_evl, 'fileids': fileids['evl']},
            open(f'{DATADIR}/{trainset}/models/{model_id}/evl.pkl', 'wb'))

print(f"Dev MAE:  {mean_absolute_error(labels_age['dev'].astype(float), hyp_dev):.3f}")
print("Predictions saved! You can now run create_submission_file() using this model_id.")


 RETRAINING & SAVING RIDGE REGRESSOR ON FULL DATA
Training on the full train set...
Final model selected alpha: 100.0
Model saved to /home/luis/projects/slp_lab2/data/train/models//ridge_baseline_combined-xvect-ecapa_2026-05-22_16:47:06/model.pkl
Model saved as: ridge_baseline_combined-xvect-ecapa_2026-05-22_16:47:06
Predicting on Dev and Evl sets...
Dev MAE:  5.445
Predictions saved! You can now run create_submission_file() using this model_id.


#### 3.1 Analyze results on the dev set and prepare your submission file

Let's check our performance on the dev set:

In [49]:
from pf_tools import save_model
import pickle
import torch
from sklearn.metrics import mean_absolute_error

# Assuming your trained model is named 'final_net'
target_model = final_net 

model_id = save_model(target_model, f'nnet_fixed_{transform_id}',
                      f'{DATADIR}/{trainset}/models/')

target_model.eval() 
with torch.no_grad():
    # THE FIX: Swapped X_dv_clean and X_ev_clean for the live data['dev'] and data['evl']
    hyp_dev = target_model(
        torch.FloatTensor(data['dev']).to(device)
    ).cpu().numpy().ravel()
    
    hyp_evl = target_model(
        torch.FloatTensor(data['evl']).to(device)
    ).cpu().numpy().ravel()

pickle.dump({'hyp': hyp_dev, 'fileids': fileids['dev']},
            open(f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl', 'wb'))
pickle.dump({'hyp': hyp_evl, 'fileids': fileids['evl']},
            open(f'{DATADIR}/{trainset}/models/{model_id}/evl.pkl', 'wb'))

print(f"Model saved as: {model_id}")
print(f"Dev MAE:  {mean_absolute_error(labels_age['dev'], hyp_dev):.2f}")

Model saved to /home/luis/projects/slp_lab2/data/train/models//nnet_fixed_combined-xvect-ecapa_2026-05-22_16:45:30/model.pkl
Model saved as: nnet_fixed_combined-xvect-ecapa_2026-05-22_16:45:30
Dev MAE:  6.27


And generate the final prediction file and make a submission to the  [Kaggle competition](https://www.kaggle.com/t/8d80747e0c474688a83024aabdfe1ab0):

In [51]:
from pf_tools import create_submission_file

students_group = '19' 

# 1. Put the exact model_id from your output here
model_id = 'ridge_baseline_combined-xvect-ecapa_2026-05-22_16:47:06'

# 2. Make a clean, short name for your final .csv file
model_id_short = 'ridge_baseline_combined'

results_path = f'{DATADIR}/{trainset}/models/{model_id}/'
filename = f'{CWD}/g{students_group}_{trainset}_{model_id_short}.csv'

create_submission_file(results_path, filename)
print(f"Submission file created successfully: {filename}")

Submission file created successfully: /home/luis/projects/slp_lab2/src/g19_train_ridge_baseline_combined.csv


At this point, you can explore different x-vector model configurations for feature extraction and alternative  neural model architectures and parameters.

# Contacts and support
You can contact the professors during the classes or the office hours.

Particularly, for this second laboratory assignment, you should contact Prof. Alberto Abad: alberto.abad@tecnico.ulisboa.pt


